# Export embeddings

Load the pretrained `outputs/model.onnx` encoder. It accepts longitude/latitude and optional time and returns embeddings; no classifier is included.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neuralearthfields/neuralearthfields-starter/blob/main/inference.ipynb)

## Setup

Locally, use the same uv environment. In Colab, run the installation cell and provide the `.onnx` file. Raster export also needs the source COG for its grid and ocean mask.

In [1]:
import os
import subprocess
import sys
import tomllib
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/neuralearthfields-starter")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/neuralearthfields/neuralearthfields-starter.git",
                str(repo),
            ],
            check=True,
        )
    os.chdir(repo)
    sys.path.insert(0, str(repo))
    dependencies = tomllib.loads(Path("pyproject.toml").read_text())["project"][
        "dependencies"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", *dependencies], check=True)

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import onnxruntime as ort
import rasterio
from rasterio.shutil import copy as raster_copy
from rasterio.windows import Window

source = Path("data/CGLC-MODIS-LCZ-100m.cog.tif")
output_dir = Path("outputs/cogs")
batch_size = 4096
write_global_cogs = False
class_codes = np.r_[1:17, 18:22, 51:62].astype("uint8")
session = ort.InferenceSession("outputs/model.onnx", providers=["CPUExecutionProvider"])

## Predict embeddings

`coords` is float32 `[batch, 2]` longitude/latitude in degrees. Optional `time` is float64 `[batch, 1]` UTC Unix seconds, or `None` when empty. This model ignores time. Any positive batch size works.

In [3]:
coords = np.array([[-122.42, 37.77], [2.35, 48.86], [36.82, -1.29]], dtype="float32")
embeddings = np.asarray(
    session.run(["embeddings"], {"coords": coords, "time": None})[0]
)
print(embeddings.shape)

(3, 128)


## Export embedding COGs

Export float32 embeddings on the source's nominal 100 m EPSG:4326 grid, with ocean and invalid pixels masked to NaN. Pass a `Window` for a subset. The full 128-band export needs about 31.5 TB before compression, plus temporary files and overviews.

In [4]:
def pixel_coordinates(transform, window):
    rows, cols = np.indices((int(window.height), int(window.width)))
    lon, lat = transform * (cols + window.col_off + 0.5, rows + window.row_off + 0.5)
    return np.column_stack((lon.ravel(), lat.ravel())).astype("float32")


def predict_tile(session, coords, valid, batch_size=4096):
    assert batch_size > 0
    bands = session.get_outputs()[0].shape[1]
    features = np.full((len(coords), bands), np.nan, dtype="float32")
    pixels = np.flatnonzero(valid.ravel())
    for start in range(0, len(pixels), batch_size):
        selected = pixels[start : start + batch_size]
        features[selected] = session.run(
            ["embeddings"], {"coords": coords[selected], "time": None}
        )[0]
    return features.T.reshape(bands, *valid.shape)


def export_embeddings(session, source, output_dir, window=None, batch_size=4096):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / "embeddings_100m.cog.tif"
    with rasterio.open(source) as src, TemporaryDirectory(dir=output_dir) as temporary:
        window = window or Window(0, 0, src.width, src.height)
        staged = Path(temporary) / "embeddings.tif"
        with rasterio.open(
            staged,
            "w",
            driver="GTiff",
            crs=src.crs,
            transform=src.window_transform(window),
            width=int(window.width),
            height=int(window.height),
            count=session.get_outputs()[0].shape[1],
            dtype="float32",
            nodata=np.nan,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            compress="zstd",
            predictor=3,
            BIGTIFF="YES",
            NUM_THREADS="4",
        ) as dst:
            for band in range(dst.count):
                dst.set_band_description(band + 1, f"embedding_{band}")
            for _, tile in dst.block_windows(1):
                source_tile = Window(
                    window.col_off + tile.col_off,
                    window.row_off + tile.row_off,
                    tile.width,
                    tile.height,
                )
                labels = src.read(1, window=source_tile, masked=True)
                valid = np.isin(labels.data, class_codes) & ~np.ma.getmaskarray(labels)
                coords = pixel_coordinates(src.transform, source_tile)
                dst.write(predict_tile(session, coords, valid, batch_size), window=tile)
        raster_copy(
            staged,
            destination,
            driver="COG",
            COMPRESS="ZSTD",
            LEVEL=3,
            BLOCKSIZE=512,
            BIGTIFF="YES",
            NUM_THREADS="4",
            PREDICTOR="FLOATING_POINT",
            OVERVIEW_RESAMPLING="AVERAGE",
        )
    return destination

In [5]:
if write_global_cogs:
    embedding_path = export_embeddings(
        session, source, output_dir, batch_size=batch_size
    )
    print(embedding_path)